<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 2: Estadística descriptiva</span><br/>
  <span style="color:DodgerBlue; font-size:140%;">Análisis exploratorio de datos </span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 1: Estadística descriptiva con Datos Reales</font>**


**Meta:** cargar datos reales de sismos e identificar:
- tipos de variables y escalas,
- tablas de frecuencia (FA, FR, FAA, FRA),
- medidas de tendencia central, posición y dispersión,
- gráficas (barras, pastel, histograma, caja y bigotes),
- análisis por grupos (clases de magnitud, horas del día).

**Fuente:** https://corgis-edu.github.io/corgis/csv/earthquakes/

In [ ]:
# 0) Importaciones (solo pandas y matplotlib)
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# 1) Cargar el dataset
url = "https://corgis-edu.github.io/corgis/datasets/csv/earthquakes/earthquakes.csv"
eq = pd.read_csv(url)

In [ ]:
eq.head()

In [ ]:
eq.tail()

In [ ]:
print("Dimensiones:", eq.shape)


**Columnas clave que usaremos (nombres reales del CSV):**
- `impact.magnitude` (float): magnitud del sismo.
- `impact.gap` (float, 0–180): calidad geométrica de estaciones (menor es mejor).
- `impact.significance` (int, 0–1000): relevancia del evento.
- `location.depth` (float, km): profundidad.
- `location.distance` (float, grados): distancia aprox. a la estación (1° ≈ 111.2 km).
- `location.name` (str): estado/país.
- `time.epoch` (int, milisegundos desde 1970-01-01).


In [ ]:
# 2) Preparación básica de tipos (cast a numérico donde aplica)
for col in ["impact.magnitude", "impact.gap", "impact.significance",
            "location.depth", "location.distance",
            "time.year", "time.month", "time.day", "time.hour", "time.minute", "time.second"]:
    if col in eq.columns:
        eq[col] = pd.to_numeric(eq[col], errors="coerce")

In [ ]:
#Vamos a ver qué tipo de varibles son cada una (en Python)
eq[["impact.magnitude", "impact.gap", "impact.significance",
            "location.depth", "location.distance",
            "time.year", "time.month", "time.day", "time.hour", "time.minute", "time.second"]].dtypes

In [ ]:
# 3) Convertiremos la distancia a (1° ≈ 111.2 km)
if "location.distance" in eq.columns:
    eq["location.distance_km"] = eq["location.distance"] * 111.2

eq["location.distance_km"]

In [ ]:
#4) Clases de magnitud (bins fijos)
# Crearemos una clasificación de los sismos:

if "impact.magnitude" in eq.columns:
    bins = [-1e9, 2, 4, 5, 6, 7, 8, 1e9]
    labels = ["Micro (<2)", "Menor [2,4)", "Ligero [4,5)", "Moderado [5,6)",
              "Fuerte [6,7)", "Mayor [7,8)", "Gran (≥8)"]
    eq["mag.class"] = pd.cut(eq["impact.magnitude"], bins=bins, labels=labels)

eq["mag.class"]

Escribe tu respuesta en este espacio y explica:

### **<font color="DodgerBlue"> Tipos de variables y escalas</font>**
<font color="DarkBlue"> ¿Cuáles son las Cualitativas nominales?
__________________________________________________
<font color="DarkBlue"> ¿Cuáles son las Cualitativas ordinales?
___________________________________________________
<font color="DarkBlue"> ¿Cuáles son las Cuantitativas continuas?
__________________________________________________


### **<font color="DodgerBlue"> TABLAS DE FRECUENCIA — mag.class (FA, FR, FAA, FRA)</font>**
FA - Frecuencia Absoluta
FR - Frecuencia Relativa
FAA - Frecuencia Absoluta Acumulada
FRA - Frecuencia Relativa Acumulada

**¿Cuál es la diferencia entre las 4 anteriores?** 
Escribe tu respuesta:

________________________________________
________________________________________
________________________________________
**Tabla de frecuencias completa** para `mag.class` (FA, FR, FAA, FRA). Describe qué clase domina y a qué FRA llegas con las 3 primeras clases.

Escribe tu respuesta:

________________________________________
________________________________________
________________________________________


In [ ]:
#Elegimos la variable con la que vamos a trabajar
serie = eq["mag.class"]
#calculo de la frecuencia absoluta
fa = serie.value_counts(dropna=False).sort_index()
#calculo de la frecuencia relativa
fr = serie.value_counts(normalize=True, dropna=False).sort_index()

#Creamos el dataframe
tabla_mag = pd.DataFrame({
    "FA": fa,
    "FR": fr
})

#Agregamos al dataframes las frecuencias acumuladas.
tabla_mag["FAA"] = tabla_mag["FA"].cumsum()
tabla_mag["FRA"] = tabla_mag["FR"].cumsum()
tabla_mag



#### **<font color="DodgerBlue"> TABLAS DE FRECUENCIA por ciudad</font>**
Escribe alguna conclución de los siguientes datas:
__________________________________________________
_________________________________________________
________________________________________________

In [ ]:
# B.2) TABLAS DE FRECUENCIA — Top 10 'location.name'
# (FA, FR, FAA, FRA sobre el subconjunto Top 10)
if "location.name" in eq.columns:
    top10 = eq["location.name"].value_counts().head(10).index
    serie_loc = eq.loc[eq["location.name"].isin(top10), "location.name"]
    fa_loc = serie_loc.value_counts().sort_index()
    fr_loc = (serie_loc.value_counts(normalize=True)).sort_index()
    tabla_loc = pd.DataFrame({"FA": fa_loc, "FR": fr_loc})
    tabla_loc["FAA"] = tabla_loc["FA"].cumsum()
    tabla_loc["FRA"] = tabla_loc["FR"].cumsum()
    display(tabla_loc)


### **<font color="DodgerBlue"> Medidas de tendencia central, posición y dispersión</font>**

Usaremos: `count`, `mean`, `median`, `mode`, `quantile`, `var`, `std`, `min`, `max`.

¿Qué significa cada uno de ellos?


In [ ]:
# C.1) impact.magnitude
col = "impact.magnitude"
s = eq[col].dropna()
res_mag = pd.Series({
    "n": s.count(),
    "media": s.mean(),
    "mediana": s.median(),
    "moda": s.mode().iloc[0] if not s.mode().empty else None,
    "q1": s.quantile(0.25),
    "q3": s.quantile(0.75),
    "p90": s.quantile(0.90),
    "var": s.var(ddof=1),
    "std": s.std(ddof=1),
    "min": s.min(),
    "max": s.max(),
    "rango": s.max() - s.min(),
    "IQR": s.quantile(0.75) - s.quantile(0.25)
})
res_mag


In [ ]:
#EJERCICIO: Ahora estudia la variable location.depth (km)
# C.2) location.depth (km)
col = "location.depth"
s = eq[col].dropna()
res_depth = pd.Series({
    "n": _________________,
    "media": _________________,
    "mediana": _________________,
    "moda": _________________
    "q1": _________________,
    "q3": _________________,
    "p90": _________________),
    "var": _________________,
    "std": _________________,
    "min": _________________,
    "max": _________________,
    "rango":_________________,
    "IQR": _________________
})
res_depth


In [ ]:
## EJERCICIO Obtén el resumen estadístico de la variable  impact.significance
# C.3) impact.significance y 
col = "impact.significance"
s = eq[col].dropna()
res_sig = pd.Series({
 ____________________
    ____________
    _____
    _____
    
})
res_sig


### Algunas consideraciones:

In [ ]:
#Un atajo que no está completo:

eq.describe()

In [ ]:
## Algo extra sobre la moda

eq['impact.magnitude'].mode()


### **<font color="DodgerBlue"> Gráficos importantes</font>**


In [ ]:
import seaborn as sns
sns.set(style="whitegrid")   # estilo más limpio y profesional

In [ ]:
tabla_mag  = tabla_mag.reset_index(drop = False)
tabla_mag

In [ ]:
# D.1) Barras por clase de magnitud
plt.figure(figsize=(8,5))
order = tabla_mag["mag.class"].value_counts().sort_index().index
sns.barplot(x=tabla_mag["mag.class"], y = tabla_mag['FR'], order=order, color="salmon")
plt.title("Número de eventos por clase de magnitud", fontsize=14)
plt.xlabel("Clase de magnitud")
plt.ylabel("Frecuencia")
plt.xticks(rotation=90)
plt.show()


## Pregunta:
## Que tipo de grafico es el de arriba?

Y ¿a qué conclusiones puedes llegar con la información de gráfico anterior?
__________________________________________________
_________________________________________________
________________________________________________

In [ ]:
# D.2) Pastel por ubicación (Top 10 'location.name')
vc = eq["location.name"].value_counts().head(15)

plt.figure(figsize=(6,6))
plt.pie(vc, labels=vc.index, autopct="%1.1f%%",startangle=180, colors=sns.color_palette("pastel"))
plt.title("Distribución de sismos por ubicación (Top 10)", fontsize=14)
plt.show()

Y ¿a qué conclusiones puedes llegar con la información de gráfico anterior?
__________________________________________________
_________________________________________________
________________________________________________

In [ ]:
# D.3) Histograma de magnitudes
plt.figure(figsize=(8,5))
sns.histplot(eq["impact.magnitude"].dropna(), binwidth=0.5, kde=True, color="skyblue", stat = 'percent') #percent
plt.title("Distribución de magnitudes sísmicas", fontsize=14)
plt.xlabel("Magnitud")
plt.ylabel("Frecuencia")
plt.show()


Y ¿a qué conclusiones puedes llegar con la información de gráfico anterior?
__________________________________________________
_________________________________________________
________________________________________________

In [ ]:
# D.4) Caja y bigotes de profundidad
plt.figure(figsize=(10,10))
sns.boxplot(x=eq["location.depth"].dropna(), color="lightgreen")
plt.title("Caja y Bigotes – Profundidad de sismos (km)", fontsize=14)
plt.ylabel("Profundidad (km)")
plt.show()


Y ¿a qué conclusiones puedes llegar con la información de gráfico anterior?
__________________________________________________
_________________________________________________
________________________________________________

In [ ]:
def HistBox(variables, df):
    # Para gráficos y visualizaciones en general
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    plt.figure(figsize=(10, 8))
    outer_grid = gridspec.GridSpec(1, 1, wspace=0.2, hspace=0.4)
    custom_params = {"axes.spines.right": True, "axes.spines.top": True, "axes.spines.left": True}
    sns.set_theme(style='whitegrid', context="talk", rc=custom_params, color_codes = True)
    sns.set_style("ticks", {"axes.grid": True, "grid.linestyle": "--"})

    for i, variable in enumerate(variables):
        inner_grid = gridspec.GridSpecFromSubplotSpec(2, 1,
                                                      subplot_spec=outer_grid[i],
                                                      height_ratios=[4, 1],
                                                      hspace=0.02)  # Ajustado para minimizar el espacio entre histogramas y boxplots

        # Histograma
        ax_hist = plt.Subplot(plt.gcf(), inner_grid[0])
        sns.histplot(df[variable], bins = 100 , kde=True, stat='count', color='yellowgreen', ax=ax_hist)
        ax_hist.set_title(variable)
        ax_hist.set_xlabel('')  # Mantener esto para limpieza
        plt.gcf().add_subplot(ax_hist)
        ax_hist.tick_params(axis='x',          # Cambios para ocultar solo los ticks (no las etiquetas, que ya están ocultas)
                            which='both',      # Afecta a ticks mayores y menores
                            bottom=False,      # Oculta ticks inferiores
                            top=False,         # Oculta ticks superiores (si estuvieran visibles)
                            labelbottom=False) # Oculta las etiquetas de los ticks inferiores
        # Calcula y anota estadísticas
        mean = df[variable].mean()
        median = df[variable].median()
        mode = df[variable].mode()[0]  # Moda
        max_value = df[variable].max()
        min_value = df[variable].min()

        # Agrega líneas verticales para cada estadística
        lines = [
            ax_hist.axvline(mean, color='red', linestyle='dashed', linewidth=2),
            ax_hist.axvline(median, color='green', linestyle='dashed', linewidth=2),
            ax_hist.axvline(mode, color='blue', linestyle='dashed', linewidth=2),
            ax_hist.axvline(max_value, color='black', linestyle='dashed', linewidth=2),
        ]

        ax_hist.grid(True)

        # Boxplot
        ax_box = plt.Subplot(plt.gcf(), inner_grid[1], sharex=ax_hist)
        sns.boxplot(x=df[variable], ax=ax_box, color='lightgreen', showmeans=True,
                    meanprops={"marker":"o", "markerfacecolor":'red',
                               "markeredgecolor":"gray", "markersize":"10"})
        ax_box.set_xlabel('')
        ax_box.tick_params(axis='x',          # Cambios para ocultar solo los ticks (no las etiquetas, que ya están ocultas)
                            which='both',      # Afecta a ticks mayores y menores
                            bottom=True,      # Oculta ticks inferiores
                            top=True,         # Oculta ticks superiores (si estuvieran visibles)
                            labelbottom=True)
        plt.gcf().add_subplot(ax_box)

In [ ]:
HistBox(['impact.magnitude'],eq)

Y ¿a qué conclusiones puedes llegar con la información de gráfico anterior?
¿Qué información nos proporcionan los dos gráficos juntos?
__________________________________________________
_________________________________________________
________________________________________________

---
<a name='ej-3'></a>
### **<font color="DodgerBlue">Ejercicio: * </font>**

<font color="DarkBlue"> Elije una variable (válida) y:

1. Muestra el resumen estadístico completo (medidas de tendencia central, dispersión y posición)
2. Realiza los gráficos correspontientes.
3. Escribe la interpretación de tus resultados.

---